# Notebook 7 : Multimodal Fusion QuantFormer

## Objective

This notebook extends the original QuantFormer architecture by
integrating Financial News embeddings generated using FinBERT with
Limit Order Book (LOB) features from the FI-2010 dataset.

The multimodal architecture combines:

- Limit Order Book Features
- Financial News Embeddings
- Fusion Layer
- Temporal Fusion Transformer

to improve short-term stock price movement prediction.

---

## Pipeline

Financial News
      │
      ▼
   FinBERT
      │
      ▼
News Embedding (768)

LOB Sequence
(100 × 143)
      │
      ▼
Input Projection
      │
      ▼
LSTM Encoder
      │
      ▼
Fusion Layer
      │
      ▼
Temporal Fusion Transformer
      │
      ▼
Classifier
      │
      ▼
Down / Stable / Up

In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project Root")

print(PROJECT_ROOT)

Project Root
d:\Coding\Quant Former


In [2]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import torch

import torch.nn as nn

from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [3]:
from src.config import *

from src.models.tft_model import TemporalFusionTransformer

from src.fusion.news_projection import NewsProjection

from src.fusion.fusion_layer import FusionLayer

from src.fusion.multimodal_dataset import MultiModalDataset

In [4]:
DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else

    "cpu"

)

print("Device :", DEVICE)

Device : cpu


In [5]:

print("Notebook 7 : Fusion QuantFormer")

print("LOB Features      : FI-2010")

print("News Embeddings   : FinBERT")

print("Fusion Strategy   : Feature-Level Fusion")

print("Prediction Task   : 3-Class Price Movement")



Notebook 7 : Fusion QuantFormer
LOB Features      : FI-2010
News Embeddings   : FinBERT
Fusion Strategy   : Feature-Level Fusion
Prediction Task   : 3-Class Price Movement


# Section 2 : Load Multimodal Data

## Objective

In this section we load both data modalities required for
Fusion QuantFormer:

- FI-2010 Limit Order Book sequences
- FinBERT News Embeddings

The loaded datasets are verified to ensure that the number of
samples matches before multimodal training.

In [6]:
LOB_DATA_PATH = "../datasets/processed/FI2010"

TRAIN_LOB = os.path.join(
    LOB_DATA_PATH,
    "X_train.npy"
)

VAL_LOB = os.path.join(
    LOB_DATA_PATH,
    "X_val.npy"
)

TEST_LOB = os.path.join(
    LOB_DATA_PATH,
    "X_test.npy"
)

TRAIN_LABELS = os.path.join(
    LOB_DATA_PATH,
    "y_train.npy"
)

VAL_LABELS = os.path.join(
    LOB_DATA_PATH,
    "y_val.npy"
)

TEST_LABELS = os.path.join(
    LOB_DATA_PATH,
    "y_test.npy"
)

TRAIN_NEWS = "../artifacts/train_finbert_embeddings.npy"

VAL_NEWS = "../artifacts/val_finbert_embeddings.npy"

TEST_NEWS = "../artifacts/test_finbert_embeddings.npy"

In [7]:
X_train = np.load(TRAIN_LOB)

X_val = np.load(VAL_LOB)

X_test = np.load(TEST_LOB)

y_train = np.load(TRAIN_LABELS)

y_val = np.load(VAL_LABELS)

y_test = np.load(TEST_LABELS)

print("LOB data loaded successfully.")

LOB data loaded successfully.


In [9]:
train_news = np.load(TRAIN_NEWS)

val_news = np.load(VAL_NEWS)

# test_news = np.load(TEST_NEWS)

print("News embeddings loaded successfully.")

News embeddings loaded successfully.


In [11]:
print("=" * 60)

print("LOB Shapes")

print("=" * 60)

print("Train :", X_train.shape)

print("Validation :", X_val.shape)

print("Test :", X_test.shape)

print()

print("=" * 60)

print("News Embedding Shapes")

print("=" * 60)

print("Train :", train_news.shape)

print("Validation :", val_news.shape)

# print("Test :", test_news.shape)

LOB Shapes
Train : (289821, 100, 143)
Validation : (72381, 100, 143)
Test : (31838, 100, 143)

News Embedding Shapes
Train : (1807, 768)
Validation : (452, 768)


In [13]:
# ==========================================================
# Validate Sample Alignment
# ==========================================================

assert len(X_train) == len(train_news) == len(y_train)

assert len(X_val) == len(val_news) == len(y_val)

assert len(X_test) == len(test_news) == len(y_test)

print("All modalities are aligned successfully.")

AssertionError: 

In [14]:
print("X_train      :", len(X_train))
print("train_news   :", len(train_news))
print("y_train      :", len(y_train))

print()

print("X_val        :", len(X_val))
print("val_news     :", len(val_news))
print("y_val        :", len(y_val))

X_train      : 289821
train_news   : 1807
y_train      : 289821

X_val        : 72381
val_news     : 452
y_val        : 72381
